In [1]:
import requests
import json
import pandas as pd
from datetime import datetime

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# GBFS Station Information API endpoint
url = "https://gbfs.bluebikes.com/gbfs/en/station_information.json"
print(f"Fetching data from: {url}")
print("Making API call...")

# Make GET request to the API
response = requests.get(url, timeout=30)

# Check if request was successful (status code 200)
response.raise_for_status()

data = response.json()

print(f"Status code: {response.status_code}")
print(f"Response received successfully!")

Fetching data from: https://gbfs.bluebikes.com/gbfs/en/station_information.json
Making API call...
Status code: 200
Response received successfully!


In [3]:
# Let's see what the response contains
print("Response structure:")
print(f"Top-level keys: {list(data.keys())}")

Response structure:
Top-level keys: ['data', 'last_updated', 'ttl', 'version']


In [5]:
# Check last updated timestamp
last_updated = data['last_updated']
last_updated_time = datetime.fromtimestamp(last_updated)
print(f"\nLast updated: {last_updated_time}")


Last updated: 2026-01-28 14:32:47


In [6]:
# Check how many stations
num_stations = len(data['data']['stations'])
print(f"Number of stations: {num_stations}")

Number of stations: 595


In [7]:
# Look at one station as example
print("\nExample station (first one):")
print(json.dumps(data['data']['stations'][0], indent=2))


Example station (first one):
{
  "lat": 42.35298248,
  "short_name": "D32077",
  "station_type": "classic",
  "has_kiosk": true,
  "region_id": "10",
  "name": "Washington St at Mina Way",
  "external_id": "2113559364250874028",
  "rental_methods": [
    "KEY",
    "CREDITCARD"
  ],
  "legacy_id": "2113559364250874028",
  "eightd_has_key_dispenser": false,
  "station_id": "2113559364250874028",
  "eightd_station_services": [],
  "capacity": 15,
  "electric_bike_surcharge_waiver": false,
  "lon": -71.17322383,
  "rental_uris": {
    "ios": "https://bos.lft.to/lastmile_qr_scan",
    "android": "https://bos.lft.to/lastmile_qr_scan"
  }
}


In [8]:
# Get the stations list
stations = data['data']['stations']

print(f"Processing {len(stations)} stations...")

Processing 595 stations...


In [9]:
# Create list to hold station records
station_records = []

# Loop through each station and extract key fields
for station in stations:
    record = {
        'station_id': station.get('station_id'),
        'station_name': station.get('name'),
        'lat': station.get('lat'),
        'lon': station.get('lon'),
        'capacity': station.get('capacity'),
        'region_id': station.get('region_id'),
        'rental_methods': ','.join(station.get('rental_methods', [])),
        'has_kiosk': station.get('has_kiosk'),
    }
    station_records.append(record)

# Convert to DataFrame
df_stations = pd.DataFrame(station_records)

print(f"Created DataFrame with {len(df_stations)} stations")
print(f"Columns: {list(df_stations.columns)}")

# Show first few rows
df_stations.head()

Created DataFrame with 595 stations
Columns: ['station_id', 'station_name', 'lat', 'lon', 'capacity', 'region_id', 'rental_methods', 'has_kiosk']


,station_id,station_name,lat,lon,capacity,region_id,rental_methods,has_kiosk
0,2113559364250874028,Washington St at Mina Way,42.352982,-71.173224,15,10,"KEY,CREDITCARD",True
1,5d17f129-af8c-4f30-bc58-6a8279052772,American Legion Hwy at Hyde Park Ave,42.272654,-71.119904,19,10,"KEY,CREDITCARD",True
2,a6d6d4b2-3666-43ad-a690-7f676bdcc3fd,Hyde Park Ave at Arlington St,42.262867,-71.121631,19,10,"KEY,CREDITCARD",True
3,f834e75d-0de8-11e7-991c-3863bb43a7d0,Hayes Square - Vine St at Moulton St,42.377022,-71.056605,19,10,"KEY,CREDITCARD",True
4,829fa5a8-cc88-4ab2-9f34-b481aaa1612e,Washington St at Bowdoin St,42.299165,-71.073459,15,10,"KEY,CREDITCARD",True


In [10]:
# Check for missing values
print("Missing values:")
print(df_stations.isnull().sum())

print("\nData types:")
print(df_stations.dtypes)

print("\nBasic statistics:")
print(df_stations.describe())

Missing values:
station_id         0
station_name       0
lat                0
lon                0
capacity           0
region_id         40
rental_methods     0
has_kiosk          0
dtype: int64

Data types:
station_id         object
station_name       object
lat               float64
lon               float64
capacity            int64
region_id          object
rental_methods     object
has_kiosk            bool
dtype: object

Basic statistics:
              lat         lon    capacity
count  595.000000  595.000000  595.000000
mean    42.359819  -71.086784   17.638655
std      0.044351    0.052277    5.358418
min     42.252287  -71.247759    9.000000
25%     42.338502  -71.118469   15.000000
50%     42.357219  -71.087386   17.000000
75%     42.379481  -71.061471   19.000000
max     42.534669  -70.870214   53.000000


In [11]:
import os

# Get base directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

# Create metadata directory
metadata_dir = os.path.join(BASE_DIR, "data", "metadata")
os.makedirs(metadata_dir, exist_ok=True)

# Add fetch timestamp
df_stations['fetched_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Save as CSV
csv_path = os.path.join(metadata_dir, "stations.csv")
df_stations.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path}")

# Save as JSON
json_path = os.path.join(metadata_dir, "stations.json")
df_stations.to_json(json_path, orient='records', indent=2)
print(f"Saved JSON: {json_path}")

# Save as Parquet (for Spark)
parquet_path = os.path.join(metadata_dir, "stations.parquet")
df_stations.to_parquet(parquet_path, index=False)
print(f"Saved Parquet: {parquet_path}")

print(f"\nAll files saved to: {metadata_dir}")

Saved CSV: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata/stations.csv
Saved JSON: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata/stations.json
Saved Parquet: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata/stations.parquet

All files saved to: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata


In [13]:
# Show upload command to run manually
gcs_path = "gs://bluebikes-demand-predictor-data/metadata/stations/"

print("Files saved locally!")
print(f"Local directory: {metadata_dir}")
print("\nTo upload to GCS, run this in your terminal:")
print("="*60)
print(f"gsutil -m cp {metadata_dir}/* {gcs_path}")
print("="*60)

Files saved locally!
Local directory: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata

To upload to GCS, run this in your terminal:
gsutil -m cp /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata/* gs://bluebikes-demand-predictor-data/metadata/stations/


In [14]:
# Summary
print("="*60)
print("STATION METADATA FETCH COMPLETE")
print("="*60)
print(f"Total stations: {len(df_stations)}")
print(f"Total capacity: {df_stations['capacity'].sum()} bikes")
print(f"Average capacity: {df_stations['capacity'].mean():.1f} bikes/station")
print(f"\nLocation Coverage:")
print(f"  Latitude: {df_stations['lat'].min():.4f} to {df_stations['lat'].max():.4f}")
print(f"  Longitude: {df_stations['lon'].min():.4f} to {df_stations['lon'].max():.4f}")
print(f"\nData saved to:")
print(f"  Local: {metadata_dir}")
print("="*60)

# Show top 10 stations
print("\nTop 10 Stations by Capacity:")
df_stations.nlargest(10, 'capacity')[['station_name', 'capacity', 'lat', 'lon']]

STATION METADATA FETCH COMPLETE
Total stations: 595
Total capacity: 10495 bikes
Average capacity: 17.6 bikes/station

Location Coverage:
  Latitude: 42.2523 to 42.5347
  Longitude: -71.2478 to -70.8702

Data saved to:
  Local: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/metadata

Top 10 Stations by Capacity:


,station_name,capacity,lat,lon
261,MIT Vassar St,53,42.355601,-71.103945
476,Lansdowne T Stop,52,42.347345,-71.100168
505,South Station - 700 Atlantic Ave,47,42.352175,-71.055547
279,Deerfield St at Commonwealth Ave,43,42.349244,-71.097282
213,Forest Hills,40,42.300923,-71.114249
358,Government Center - Cambridge St at Court St,40,42.359825,-71.059796
37,Beacon St at Charles St,39,42.356052,-71.069849
184,Boston Convention and Exhibition Center - Summ...,39,42.347763,-71.045360
312,Christian Science Plaza - Massachusetts Ave at...,39,42.343666,-71.085824
273,Nashua Street at Red Auerbach Way,37,42.365673,-71.064263
